## **Imports**

In [1]:
import sys
import os

# Add the Marcelo directory to sys.path to allow merging of 'src' namespace packages.
# This ensures that Python sees both the root 'src' (contract) and 'Marcelo/src' (implementation).
marcelo_dir = os.path.abspath("Marcelo")
if marcelo_dir not in sys.path:
    sys.path.append(marcelo_dir)

from src.millionaire_client import AuthenticationError, MillionaireClient
from src.benchmark import Benchmark
from dotenv import load_dotenv
from src.models import ExperimentConfig, ApproachType
from src.guesser.marcelo_guesser import MarceloGuesser

## **Auth**

In [2]:
load_dotenv(dotenv_path=os.path.join("Marcelo", ".env"))


In [3]:
API_URL = "http://131.175.15.22:51111/"

USERNAME = os.getenv("MILLIONAIRE_USERNAME")
PASSWORD = os.getenv("MILLIONAIRE_PASSWORD")


In [4]:
client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")

# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


Welcome, MTKY! (Role: student)

=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)


In [5]:
from src.guesser.configs import INFERENCE_MODEL, EMBEDDING_MODEL

marcelo_experiment_config = ExperimentConfig(
    username="Marcelo",
    notes="Multi-theme dynamic approach test",
    approach=ApproachType.HYBRID, # Guesser will handle per-theme logic
    inference_model=INFERENCE_MODEL,
    inference_model_size=2,
    embedding_model=EMBEDDING_MODEL,
    embedding_model_size=0.2,
    is_rag=True
)

guesser = MarceloGuesser(
    marcelo_experiment_config, 
    embedding_model_name=EMBEDDING_MODEL, 
    inference_model_name=INFERENCE_MODEL
)

In [6]:
benchmark = Benchmark(marcelo_experiment_config, guesser, client)
benchmark.run(1, filename="marcelo_benchmark_results.xlsx")